# Analysis — `n1m5_T60_obs5000_seed42`

Compare the **credulous** and **vigilant** listener under two speakers (informative `inf`, persuasive `persp`), across nine true thetas. The notebook is factorized into four explicit sections:

1. **Load** the four `(speaker × listener)` belief Datasets.
2. **Compute** four per-trajectory quantities (full posterior, expected θ, MAP θ distribution, JS divergence).
3. **Aggregate** scalar summaries for line plots (central tendency + 95% interval).
4. **Visualize** with six figures:
    - **4.1** Mean posterior heatmap — one figure per speaker.
    - **4.2** Expected θ — *mean* across trajectories with 95% CI (line plot).
    - **4.3** Expected θ — *median* across trajectories with 95% CI (line plot).
    - **4.4** MAP θ distribution heatmap — one figure per speaker.
    - **4.5** JS divergence line plot — median + 95% CI.
    - **4.6** JS distribution heatmap — one figure per speaker.

Round 0 in every plot is the synthetic uniform prior, prepended to the recorded rounds 0..T-1.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Bootstrap repo root onto sys.path so absolute imports work from a notebook.
HERE = Path.cwd().resolve()
# analyze.ipynb -> n1m5.../ -> simulation_experiments/ -> simulations/ -> models/ -> repo root
REPO_ROOT = HERE.parents[3]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from models.simulations.simulation_experiments.n1m5_T60_obs5000_seed42.io import load_beliefs

DATA_ROOT = HERE / 'raw_do_not_track'
print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT.is_dir(), f'expected experiment data at {DATA_ROOT}'

In [ ]:
# Identifiers — must match the directory names produced by run.py.
SPEAKERS = {
    'inf':   'inf_L1strat_a3_b1_uiF',
    'persp': 'persp_L1strat_a3_b0_uiF',
}
LISTENERS = {
    'credulous': 'credulous_L1coop_a3_uiF',
    'vigilant':  'vigilant_L1strat_a3_uiF',
}

# TRUE_THETAS is the set of true thetas at which observations were sampled
# (the figure columns). The agents' belief grid is wider — it also includes
# 0.0 and 1.0 — and shows up in the loaded data as belief_ds.theta
# (length 11). Don't confuse the two.
TRUE_THETAS = [round(0.1 * k, 1) for k in range(1, 10)]   # [0.1, ..., 0.9]

# Plot config
LISTENER_COLORS = {'credulous': 'tab:blue', 'vigilant': 'tab:orange'}
LISTENER_TITLES = {'credulous': 'Credulous listener', 'vigilant': 'Vigilant listener'}
SPEAKER_TITLES  = {'inf': 'Informative speaker', 'persp': 'Persuasive (pers+) speaker'}

## 1. Load

In [ ]:
def load_belief_grid(data_root, speakers, listeners):
    '''Return a dict {(spk_key, lst_key): xr.Dataset} for every (speaker, listener) pair.'''
    out = {}
    for spk_key, spk_dir in speakers.items():
        for lst_key, lst_dir in listeners.items():
            out[(spk_key, lst_key)] = load_beliefs(data_root / spk_dir / lst_dir)
    return out

beliefs = load_belief_grid(DATA_ROOT, SPEAKERS, LISTENERS)
for k, ds in beliefs.items():
    print(f'{k}: sizes={dict(ds.sizes)} path={ds.attrs["execution_path"]}')

## 2. Compute

**Per-trajectory quantities** — each shape `(n_true_theta, n_traj, n_rounds)`, with round 0 = synthetic uniform prior:
- Expected θ:    $\mathbb{E}[\theta \mid u_{0..t}] = \sum_\theta \theta \cdot P(\theta \mid u_{0..t})$.
- JS divergence: $\mathrm{JS}(p_i, \bar p)$, in nats.

**Per-round summaries across trajectories** — each shape `(n_true_theta, n_rounds, n_grid)`:
- Mean posterior:   $\bar p_k(\theta)$.
- MAP distribution: tie-aware fraction of trajectories whose MAP is at each grid cell.

**JS distribution heatmap input** — bin per-trajectory JS values per round into `n_bins`, giving shape `(n_true_theta, n_rounds, n_bins)`.

`obs_idx × utt_idx` are stacked into a single trajectory axis before any of the above.

In [ ]:
def belief_with_prior(belief_ds):
    '''Stack obs+utt into traj, prepend uniform prior at round 0.'''
    bt = belief_ds.belief_theta.stack(traj=('obs_idx', 'utt_idx')).transpose(
        'theta_true', 'traj', 't', 'theta'
    )
    arr = bt.values
    n_thetas, n_traj, _, n_grid = arr.shape
    prior_block = np.full((n_thetas, n_traj, 1, n_grid), 1.0 / n_grid)
    return np.concatenate([prior_block, arr], axis=2)


def expectation_per_traj(belief_arr, theta_grid):
    '''E[θ | u_{0..t}] per trajectory; shape (n_thetas, n_traj, n_rounds).'''
    return (belief_arr * theta_grid).sum(axis=-1)


def map_distribution(belief_arr):
    '''Tie-aware empirical distribution of trajectory MAPs.

    A trajectory whose posterior has k tied modes contributes 1/k to each
    tied grid cell. Round 0's uniform prior thus spreads to 1/|Θ| per cell
    instead of collapsing to argmax-returns-zero.

    Returns shape (n_thetas, n_rounds, n_grid).
    '''
    max_val = belief_arr.max(axis=-1, keepdims=True)
    is_max = belief_arr == max_val
    n_ties = is_max.sum(axis=-1, keepdims=True)
    weights = is_max.astype(np.float64) / n_ties
    return weights.mean(axis=1)


def _kl_safe(p, q, axis=-1):
    '''KL(p || q) summed along axis; treats 0 * log(0/_) as 0 and drops infs.'''
    with np.errstate(divide='ignore', invalid='ignore'):
        terms = p * (np.log(p) - np.log(q))
    return np.where(np.isfinite(terms), terms, 0.0).sum(axis=axis)


def js_per_trajectory(belief_arr):
    '''Per-trajectory JS(p_i, mean_p) in nats; shape (n_thetas, n_traj, n_rounds).'''
    mean_p = belief_arr.mean(axis=1, keepdims=True)
    m = 0.5 * (belief_arr + mean_p)
    return 0.5 * (_kl_safe(belief_arr, m) + _kl_safe(mean_p, m))


def histogram_per_round(values_arr, bins):
    '''Histogram along the trajectory axis at each (true_theta, round).

    values_arr: (n_thetas, n_traj, n_rounds)
    bins:       1-D array of bin edges, length n_bins+1
    Returns:    (n_thetas, n_rounds, n_bins) — fraction of trajectories per bin
    '''
    n_thetas, n_traj, n_rounds = values_arr.shape
    n_bins = len(bins) - 1
    out = np.zeros((n_thetas, n_rounds, n_bins))
    for tt in range(n_thetas):
        for r in range(n_rounds):
            counts, _ = np.histogram(values_arr[tt, :, r], bins=bins)
            out[tt, r] = counts / n_traj
    return out

In [ ]:
THETA_GRID = beliefs[('inf', 'credulous')].theta.values

# Full per-trajectory posterior, with prior prepended at round 0.
full_belief = {k: belief_with_prior(ds) for k, ds in beliefs.items()}

# Cross-trajectory mean posterior (for the §4.1 heatmap).
mean_posterior = {k: arr.mean(axis=1) for k, arr in full_belief.items()}

# Per-trajectory expected θ.
e_theta_traj = {k: expectation_per_traj(arr, THETA_GRID) for k, arr in full_belief.items()}

# MAP distribution across trajectories (tie-aware).
map_dist = {k: map_distribution(arr) for k, arr in full_belief.items()}

# Per-trajectory JS divergence (nats).
js_traj = {k: js_per_trajectory(arr) for k, arr in full_belief.items()}

# Bin the per-trajectory JS into a per-round histogram (for the §4.6 heatmap).
JS_VMAX_TRAJ = max(arr.max() for arr in js_traj.values())
JS_BIN_EDGES = np.linspace(0.0, JS_VMAX_TRAJ * 1.001, 31)        # 30 bins
JS_BIN_CENTERS = 0.5 * (JS_BIN_EDGES[:-1] + JS_BIN_EDGES[1:])
js_dist = {k: histogram_per_round(arr, JS_BIN_EDGES) for k, arr in js_traj.items()}

print('Per-trajectory quantities:')
for k in beliefs:
    print(f'  {k}: e_theta {e_theta_traj[k].shape}; js {js_traj[k].shape}')
print(f'JS bins: {len(JS_BIN_CENTERS)} bins on [0, {JS_BIN_EDGES[-1]:.4f}] nats')

## 3. Aggregate

For the line plots, summarize each per-trajectory array as central tendency + 95% interval (2.5/97.5 percentiles):

- `summaries_e_theta_mean` — *mean* of E[θ] across trajectories.
- `summaries_e_theta_med`  — *median* of E[θ] across trajectories.
- `summaries_js`           — median of per-trajectory JS across trajectories.

Each summary is a tuple `(central, lo, hi)` of shape `(n_true_theta, n_rounds)`.

In [ ]:
def summarize_traj(arr, center='median', lo_pct=2.5, hi_pct=97.5):
    '''arr: (n_thetas, n_traj, n_rounds). Returns (central, lo, hi), each (n_thetas, n_rounds).'''
    if center == 'median':
        central = np.median(arr, axis=1)
    elif center == 'mean':
        central = np.mean(arr, axis=1)
    else:
        raise ValueError(f"center must be 'median' or 'mean', got {center!r}")
    lo = np.percentile(arr, lo_pct, axis=1)
    hi = np.percentile(arr, hi_pct, axis=1)
    return central, lo, hi


summaries_e_theta_mean = {k: summarize_traj(arr, center='mean')   for k, arr in e_theta_traj.items()}
summaries_e_theta_med  = {k: summarize_traj(arr, center='median') for k, arr in e_theta_traj.items()}
summaries_js           = {k: summarize_traj(arr, center='median') for k, arr in js_traj.items()}

# Quick sanity at theta_true=0.5 (idx 4)
print('Persuasive speaker, theta_true=0.5:')
for label, summ in [('mean   E[θ]', summaries_e_theta_mean),
                     ('median E[θ]', summaries_e_theta_med),
                     ('median JS  ', summaries_js)]:
    c, lo, hi = summ[('persp', 'vigilant')]
    print(f'  vigilant {label}  round 0: c={c[4, 0]:.4f}  CI=[{lo[4, 0]:.4f}, {hi[4, 0]:.4f}]')
    print(f'  vigilant {label}  round T: c={c[4, -1]:.4f}  CI=[{lo[4, -1]:.4f}, {hi[4, -1]:.4f}]')

## 4. Visualize

Two helpers used throughout:

- `plot_belief_grid` — line-plot grid (rows = speakers, cols = true θ) with listener lines + 95% CI shading.
- `plot_heatmap_grid` — heatmap grid (rows = listeners, cols = true θ); one figure per speaker.

The heatmaps use a custom diverging blue–white–red colormap (`LIGHT_BWR`) built from matplotlib's named `lightskyblue` and `lightcoral`. Where the data has a meaningful "prior" (mean posterior, MAP distribution), white is anchored at $1/|\Theta| \approx 0.091$ via `TwoSlopeNorm`. For the JS distribution heatmap (no prior), white sits at the cmap midpoint via plain `Normalize`.

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm


# Light diverging blue-white-red.
LIGHT_BWR = LinearSegmentedColormap.from_list(
    'light_bwr',
    [mcolors.to_rgb('lightskyblue'),
     (1.0, 1.0, 1.0),
     mcolors.to_rgb('lightcoral')],
    N=256,
)


def plot_belief_grid(summaries, true_thetas, speakers, listeners,
                     listener_colors, speaker_titles,
                     y_label, fig_title=None,
                     ylim=(0, 1), truth_line=True):
    '''Line-plot grid: rows = speakers, cols = true θ, listener lines overlaid.'''
    n_rows, n_cols = len(speakers), len(true_thetas)
    fig, axes = plt.subplots(n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True, squeeze=False)

    any_central = next(iter(summaries.values()))[0]
    n_rounds = any_central.shape[1]
    rounds = np.arange(n_rounds)

    for row, spk in enumerate(speakers):
        for col, theta_true in enumerate(true_thetas):
            ax = axes[row, col]
            if truth_line:
                ax.axhline(theta_true, color='black', linestyle='--',
                           linewidth=0.8, alpha=0.6)
            for lst in listeners:
                c, lo, hi = summaries[(spk, lst)]
                color = listener_colors[lst]
                ax.plot(rounds, c[col], color=color, label=lst, linewidth=1.6)
                ax.fill_between(rounds, lo[col], hi[col], color=color, alpha=0.18)
            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk]}\n{y_label}', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')
            if ylim is not None:
                ax.set_ylim(*ylim)
            ax.set_xlim(0, n_rounds - 1)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if fig_title is not None:
        fig.suptitle(fig_title, fontsize=12, y=1.02)
    fig.legend(handles, labels, loc='upper center', ncol=len(labels),
               bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


def plot_heatmap_grid(data_per_listener, y_values, true_thetas,
                      listeners, listener_titles,
                      vmax, value_label, speaker_label,
                      vcenter=None, cmap=LIGHT_BWR,
                      y_label='θ', truth_line=True):
    '''Heatmap grid: rows = listeners, cols = true θ. One figure per speaker.

    `data_per_listener[lst]`: (n_true_theta, n_rounds, n_y_bins)
    `y_values`:               1-D y-axis values (e.g. θ grid or JS bin centers)
    If `vcenter` is given, white anchored there via TwoSlopeNorm; else linear Normalize.
    '''
    if vcenter is not None:
        norm = TwoSlopeNorm(vcenter=vcenter, vmin=0.0,
                            vmax=max(vmax, vcenter * 1.001))
    else:
        norm = Normalize(vmin=0.0, vmax=vmax)

    n_rows, n_cols = len(listeners), len(true_thetas)
    fig, axes = plt.subplots(n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True, squeeze=False)
    n_rounds = next(iter(data_per_listener.values())).shape[1]

    # y-axis extent — bracket the bin centers by half a step
    if len(y_values) > 1:
        dy_lo = float(y_values[1] - y_values[0]) * 0.5
        dy_hi = float(y_values[-1] - y_values[-2]) * 0.5
    else:
        dy_lo = dy_hi = 0.05
    y_min = float(y_values[0])  - dy_lo
    y_max = float(y_values[-1]) + dy_hi

    im = None
    for row, lst in enumerate(listeners):
        data = data_per_listener[lst]
        for col, theta_true in enumerate(true_thetas):
            ax = axes[row, col]
            arr = data[col].T   # (n_y_bins, n_rounds)
            im = ax.imshow(arr,
                aspect='auto', origin='lower',
                extent=[-0.5, n_rounds - 0.5, y_min, y_max],
                norm=norm, cmap=cmap)
            if truth_line:
                ax.axhline(theta_true, color='black', linestyle='--',
                           linewidth=0.9, alpha=0.85)
            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{listener_titles[lst]}\n{y_label}', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')

    if im is not None:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(),
                            shrink=0.85, pad=0.02, aspect=25)
        cbar.set_label(value_label)
    fig.suptitle(speaker_label, fontsize=12, y=1.0)
    return fig

### 4.1 Mean posterior heatmap

Color = mean $P(\theta \mid u_{0..t})$ across trajectories. White anchored at the uniform prior $1/|\Theta| \approx 0.091$ via `TwoSlopeNorm`. Two figures (one per speaker), each 2 × 9 (rows = listeners, cols = true θ). Black dashed line marks the truth.

In [ ]:
# Shared vmax across both speakers and both listeners.
MEAN_POST_VMAX = max(arr.max() for arr in mean_posterior.values())
print(f'MEAN_POST_VMAX = {MEAN_POST_VMAX:.3f}')

In [ ]:
inf_means = {lst: mean_posterior[('inf', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    inf_means, THETA_GRID, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=MEAN_POST_VMAX,
    value_label=r'mean $P(\theta \mid u_{0..t})$',
    speaker_label='Informative speaker — mean posterior heatmap',
    vcenter=1 / len(THETA_GRID),
)
plt.show()

In [ ]:
persp_means = {lst: mean_posterior[('persp', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    persp_means, THETA_GRID, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=MEAN_POST_VMAX,
    value_label=r'mean $P(\theta \mid u_{0..t})$',
    speaker_label='Persuasive (pers+) speaker — mean posterior heatmap',
    vcenter=1 / len(THETA_GRID),
)
plt.show()

### 4.2 Expected θ — mean across trajectories

Mean of $\mathbb{E}[\theta \mid u_{0..t}]$ across trajectories with 2.5/97.5 percentile shading. Truth dashed black.

In [ ]:
plot_belief_grid(
    summaries_e_theta_mean, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label=r'mean $\mathbb{E}[\theta]$',
    fig_title='Mean of expected θ across trajectories (95% CI)',
)
plt.show()

### 4.3 Expected θ — median across trajectories

Median of $\mathbb{E}[\theta \mid u_{0..t}]$ across trajectories with 2.5/97.5 percentile shading. Truth dashed black.

In [ ]:
plot_belief_grid(
    summaries_e_theta_med, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label=r'median $\mathbb{E}[\theta]$',
    fig_title='Median of expected θ across trajectories (95% CI)',
)
plt.show()

### 4.4 MAP θ distribution heatmap

Color = fraction of trajectories whose posterior MAP is at each grid value. Tie-aware (round 0's uniform prior spreads to $1/|\Theta|$ rather than collapsing to argmax-returns-zero). Same 2 × 9 layout as §4.1, one figure per speaker.

In [ ]:
MAP_VMAX = max(arr.max() for arr in map_dist.values())
print(f'MAP_VMAX = {MAP_VMAX:.3f}')

In [ ]:
inf_maps = {lst: map_dist[('inf', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    inf_maps, THETA_GRID, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=MAP_VMAX,
    value_label='fraction of trajectories with MAP at θ',
    speaker_label='Informative speaker — MAP distribution heatmap',
    vcenter=1 / len(THETA_GRID),
)
plt.show()

In [ ]:
persp_maps = {lst: map_dist[('persp', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    persp_maps, THETA_GRID, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=MAP_VMAX,
    value_label='fraction of trajectories with MAP at θ',
    speaker_label='Persuasive (pers+) speaker — MAP distribution heatmap',
    vcenter=1 / len(THETA_GRID),
)
plt.show()

### 4.5 JS divergence — line plot

Median of per-trajectory $\mathrm{JS}(p_i, \bar p)$ in nats with 2.5/97.5 percentile shading. $D_0 = 0$ at round 0 because every trajectory shares the prior.

In [ ]:
plot_belief_grid(
    summaries_js, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label='JS (nats)',
    fig_title='JS divergence: median across trajectories (95% CI)',
    ylim=None,           # let y-axis auto-scale
    truth_line=False,
)
plt.show()

### 4.6 JS distribution heatmap

Color = fraction of trajectories whose JS divergence at round $t$ falls in a given bin (per-round histogram of `js_traj`, 30 bins on $[0, \mathrm{JS}_{\max}]$). Same 2 × 9 layout as §4.4, one figure per speaker. No prior anchor — `Normalize(0, vmax)` with white at the cmap midpoint.

In [ ]:
JS_DIST_VMAX = max(arr.max() for arr in js_dist.values())
print(f'JS_DIST_VMAX = {JS_DIST_VMAX:.3f}')

In [ ]:
inf_js = {lst: js_dist[('inf', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    inf_js, JS_BIN_CENTERS, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=JS_DIST_VMAX,
    value_label='fraction of trajectories per JS bin',
    speaker_label='Informative speaker — JS distribution heatmap',
    y_label='JS (nats)',
    truth_line=False,
)
plt.show()

In [ ]:
persp_js = {lst: js_dist[('persp', lst)] for lst in LISTENERS}
plot_heatmap_grid(
    persp_js, JS_BIN_CENTERS, TRUE_THETAS,
    listeners=list(LISTENERS.keys()),
    listener_titles=LISTENER_TITLES,
    vmax=JS_DIST_VMAX,
    value_label='fraction of trajectories per JS bin',
    speaker_label='Persuasive (pers+) speaker — JS distribution heatmap',
    y_label='JS (nats)',
    truth_line=False,
)
plt.show()

## What to look for

- **Row 1 of each heatmap (informative speaker)** — both listeners should track the truth; mass concentrates around $\theta_{true}$ over rounds.
- **Row 2 (persuasive speaker)** — credulous gets misled (mass shifts upward); vigilant compensates.
- **Mean vs median of E[θ]** (§4.2 vs §4.3) — divergence indicates a skewed trajectory distribution. Means are sensitive to outlier-trajectory tails; medians are robust.
- **MAP distribution (§4.4)** catches multimodality scalar summaries hide — e.g., a column with mass split between two cells means trajectories are converging to different MAPs.
- **JS line plot (§4.5)** shows when trajectories agree (low JS) or diverge (high JS) on average.
- **JS distribution heatmap (§4.6)** shows the *shape* of trajectory disagreement — a tight horizontal stripe means most trajectories have similar JS at that round; a wide column means trajectories differ widely in their distance from the mean.